In [ ]:
# link google derive drive
from google.colab import drive
drive.mount('/content/drive')
share_path = "/content/drive/MyDrive/Colab_Notebooks/deeplearning2026"
import os
print(os.listdir(share_path))

Mounted at /content/drive
['Siavash', 'NLP-Arch.gdoc', 'documentation', 'notebooks', 'data']


In [2]:
# install needed packages
!pip install sigfig


In [3]:
# prepare environment
import pandas as pd
import numpy as np
from sigfig import round

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.min_rows', 30)


In [4]:
# load data
data_path = share_path + "/data/"

inv = pd.read_csv(data_path + 'fndds_ingredient_nutrient_value.csv',
                  usecols = [1,2,3,5],
                  dtype = {'Ingredient description': 'str',
                           'nutrient code': np.float64,
                           'nutrient value': np.float64,
                           'FDC ID': 'str'},
                  header = 0,
                  names = ['Ingredient_description',
                           'nutrient_nbr','nutrient_value',
                           'fdc_id'])


# csv crosswalk of nutrient names and codes
nut = pd.read_csv(data_path + 'nutrient.csv',
                  dtype = {'name': 'str',
                           'unit_name': 'str',
                           'nutrient_nbr': np.float64,
                           'id': 'str'},
                  header = 0,
                  names = ['nutrient_id','name','unit_name',
                           'nutrient_nbr','rank'])
nut = nut.drop('rank',axis = 1)


# list of all foods, including branded foods
food = pd.read_csv(data_path + 'food.csv',
                 usecols = ['fdc_id','description',
                            'food_category_id',
                            'data_type'],
                 dtype = {
                     'fdc_id': 'str',
                     'description': 'str',
                     'food_category_id': 'str',
                     'data_type': 'str'
                 })

fn = pd.read_csv(data_path + 'food_nutrient.csv',
                 dtype = {'fdc_id': 'str',
                          'nutrient_id': 'str',
                          'amount': np.float64},
                 header = 0,
                 names = ['id','fdc_id','nutrient_id',
                          'nutrient_value','dp','d','min','max',
                          'med','loq','foot','yr','pdv'])
fn = fn.drop(['id','dp','d','min','max','med','loq','foot','yr','pdv'],
             axis = 1)


/tmp/ipykernel_1103/2344432238.py:40: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  fn = pd.read_csv(data_path + 'food_nutrient.csv',


In [5]:
# merge
nut['nutrient_nbr'] = nut['nutrient_nbr'].astype(np.float64)
inv['nutrient_nbr'] = inv['nutrient_nbr'].astype(np.float64)
inv = pd.merge(inv, nut, how = 'right', on = 'nutrient_nbr')
nut['nutrient_id'] = nut['nutrient_id'].astype('str')
fn['nutrient_id'] = fn['nutrient_id'].astype('str')
fn = fn[fn['nutrient_id'].isin(nut['nutrient_id'])]
fn = pd.merge(fn, nut, how = 'right', on = 'nutrient_id')
fn['Ingredient_description'] = np.nan
fn = pd.concat([fn,inv])

del inv, nut

In [6]:
fn['fdc_id'] = fn['fdc_id'].astype(np.float64)
food['fdc_id'] = food['fdc_id'].astype(np.float64)
df = pd.merge(food, fn, how = 'right', on = 'fdc_id')
print(df.shape)

del food, fn

(27370184, 10)


In [7]:
df = df.dropna(subset = ['nutrient_value'])
df = df.drop(['nutrient_id', 'nutrient_nbr'], axis = 1)
print(df.shape)


(27369563, 8)


In [8]:
df.loc[df['description'].isna(), 'description'] = df.loc[df['description'].isna(), 'Ingredient_description']
df.drop(['Ingredient_description'], axis = 1, inplace = True)

In [9]:
# nutrients of interest
noi = ['Energy (Atwater General Factors)',
       'Energy (Atwater Specific Factors)',
       'Protein','Total lipid (fat)',
       'Carbohydrate, by difference',
       'Energy', 'Starch', 'Sucrose',
       'Glucose', 'Fructose', 'Lactose',
       'Maltose', 'Alcohol, ethyl',
       'Carbohydrate, by summation',
       'Water','Caffeine','Sugars, Total',
       'Fiber, total dietary','Fiber, soluble',
       'Fiber, insoluble', 'Total fat (NLEA)',
       'Calcium, Ca', 'Chlorine, Cl', 'Iron, Fe', 'Magnesium, Mg',
       'Phosphorus, P', 'Potassium, K', 'Sodium, Na', 'Sulfur, S',
       'Zinc, Zn', 'Chromium, Cr', 'Cobalt, Co', 'Copper, Cu',
       'Fluoride, F', 'Iodine, I', 'Manganese, Mn', 'Molybdenum, Mo',
       'Selenium, Se', 'Vitamin A, IU',
       'Vitamin E (alpha-tocopherol)',
       'Vitamin D (D2 + D3), International Units',
       'Vitamin D (D2 + D3)','Lycopene',
       'Boron, B', 'Nickel, Ni', 'Vitamin E',
       'Vitamin C, total ascorbic acid', 'Thiamin', 'Riboflavin',
       'Niacin', 'Pantothenic acid', 'Vitamin B-6', 'Biotin',
       'Folate, total', 'Vitamin B-12', 'Choline, total',
       'Vitamin K (Menaquinone-4)', 'Vitamin K (Dihydrophylloquinone)',
       'Vitamin K (phylloquinone)',
       'Tryptophan', 'Threonine',
       'Isoleucine', 'Leucine', 'Lysine', 'Methionine', 'Cystine',
       'Phenylalanine', 'Tyrosine', 'Valine', 'Arginine', 'Histidine',
       'Alanine', 'Aspartic acid', 'Glutamic acid', 'Glycine', 'Proline',
       'Serine', 'Hydroxyproline', 'Cysteine', 'Taurine',
       'Cholesterol', 'Fatty acids, total trans',
       'Fatty acids, total saturated', 'PUFA 20:4',
       'PUFA 22:6 n-3 (DHA)','PUFA 20:5 n-3 (EPA)',
       'PUFA 22:5 n-3 (DPA)','Fatty acids, total monounsaturated',
       'Fatty acids, total polyunsaturated',
       'PUFA 20:2 n-6 c,c','PUFA 18:2 n-6 c,c',
       'PUFA 18:3 n-6 c,c,c',
       'PUFA 18:3 n-3 c,c,c (ALA)', 'PUFA 20:3 n-3', 'PUFA 20:3 n-6',
       'PUFA 20:4 n-6','Total dietary fiber (AOAC 2011.25)']

df = df[df['name'].isin(noi)]
print(df.shape)
del noi

(24342171, 7)


In [10]:
rename = {'name': {'Total lipid (fat)': 'total_fat',
                   'Vitamin E (alpha-tocopherol)': 'alpha_tocopherol',
                   'Vitamin E (label entry primarily)': 'label_vit_e',
                   'Vitamin E': 'vit_e',
                   'Choline, total': 'total_choline',
                   'Choline, free': 'free_choline',
                   'Choline, from phosphocholine': 'phosphocholine',
                   'Choline, from phosphotidyl choline': 'phosphotidylcholine',
                   'Choline, from glycerophosphocholine': 'glycerophosphocholine',
                   'Choline, from sphingomyelin': 'sphingomyelin',
                   'Vitamin E, added': 'added_vit_e',
                   'Cholesterol': 'cholesterol',
                   'Fatty acids, total saturated': 'sat_fat',
                   'PUFA 18:2': 'LA',
                   'PUFA 18:3': 'pufa_18_3',
                   'PUFA 22:6 n-3 (DHA)': 'DHA',
                   'PUFA 20:5 n-3 (EPA)': 'EPA',
                   'PUFA 22:5 n-3 (DPA)': 'DPA',
                   'Fatty acids, total monounsaturated': 'MUFA',
                   'Fatty acids, total polyunsaturated': 'PUFA',
                   'PUFA 18:2 n-6 c,c': 'pufa_18_2_n6',
                   'PUFA 20:2 n-6 c,c': 'pufa_20_2_n6',
                   'PUFA 20:4': 'pufa_20_4',
                   'PUFA 18:3 n-6 c,c,c': 'pufa_18_3_n6',
                   'PUFA 18:3 n-3 c,c,c (ALA)': 'ALA',
                   'PUFA 20:3 n-3': 'pufa_20_3_n3',
                   'PUFA 20:3 n-6': 'pufa_20_3_n6',
                   'PUFA 20:4 n-6': 'ARA',
                   'Energy (Atwater General Factors)': 'kcal1',
                   'Energy (Atwater Specific Factors)': 'kcal2',
                   'Protein': 'protein',
                   'Carbohydrate, by difference': 'CHO1',
                   'Energy': 'kcal3',
                   'Starch': 'CHOstarch',
                   'Sucrose': 'sucrose',
                   'Glucose': 'glucose',
                   'Fructose': 'fructose',
                   'Lactose': 'lactose',
                   'Maltose': 'maltose',
                   'Alcohol, ethyl': 'alcohol',
                   'Carbohydrate, by summation': 'CHO2',
                   'Water': 'water',
                   'Caffeine': 'caffeine',
                   'Sugars, Total': 'CHOsugar',
                   'Fiber, total dietary': 'CHOfiber1',
                   'Fiber, soluble': 'sol_fiber',
                   'Fiber, insoluble': 'insol_fiber',
                   'Total fat (NLEA)': 'total_fat2',
                   'Calcium, Ca': 'Ca',
                   'Chlorine, Cl': 'Cl',
                   'Iron, Fe': 'Fe',
                   'Magnesium, Mg': 'Mg',
                   'Phosphorus, P': 'P',
                   'Potassium, K': 'K',
                   'Sodium, Na': 'Na',
                   'Sulfur, S': 'S',
                   'Zinc, Zn': 'Zn',
                   'Chromium, Cr': 'Cr',
                   'Cobalt, Co': 'Co',
                   'Copper, Cu': 'Cu',
                   'Fluoride, F': 'F',
                   'Iodine, I': 'I',
                   'Manganese, Mn': 'Mn',
                   'Molybdenum, Mo': 'Mo',
                   'Selenium, Se': 'Se',
                   'Vitamin A, IU': 'vit_a',
                   'Vitamin D (D2 + D3), International Units': 'total_vit_d',
                   'Vitamin D (D2 + D3)': 'total_vit_d',
                   'Lycopene': 'lycopene',
                   'Boron, B': 'B',
                   'Nickel, Ni': 'Ni',
                   'Vitamin C, total ascorbic acid': 'vit_c',
                   'Thiamin': 'B1',
                   'Riboflavin': 'B2',
                   'Niacin': 'B3',
                   'Pantothenic acid': 'B5',
                   'Vitamin B-6': 'B6',
                   'Biotin': 'B7',
                   'Folate, total': 'B9',
                   'Vitamin B-12': 'B12',
                   'Vitamin K (Menaquinone-4)': 'vit_k_m',
                   'Vitamin K (Dihydrophylloquinone)': 'vit_k_d',
                   'Vitamin K (phylloquinone)': 'vit_k_p',
                   'Tryptophan': 'tryptophan',
                   'Threonine': 'threonine',
                   'Isoleucine': 'isoleucine',
                   'Leucine': 'leucine',
                   'Lysine': 'lysine',
                   'Methionine': 'methionine',
                   'Cystine': 'cystine',
                   'Phenylalanine': 'phenylalanine',
                   'Tyrosine': 'tyrosine',
                   'Valine': 'valine',
                   'Arginine': 'arginine',
                   'Histidine': 'histidine',
                   'Alanine': 'alanine',
                   'Aspartic acid': 'aspartate',
                   'Glutamic acid': 'glutamate',
                   'Glycine': 'glycine',
                   'Proline': 'proline',
                   'Serine': 'serine',
                   'Hydroxyproline': 'hydroxyproline',
                   'Cysteine': 'cysteine',
                   'Taurine': 'taurine',
                   'Fatty acids, total trans': 'trans_fat',
                   'Total dietary fiber (AOAC 2011.25)': 'CHOfiber2'
                  }}

df = df.replace(rename)


In [11]:
# get rid of duplicates so we can pivot
df.drop_duplicates(inplace = True)
# the remaining duplicates have slight differences in 'nutrient_value'
# or they have different units
df[df.duplicated(subset=['fdc_id','description','name'],keep=False)].sort_values(by = ['fdc_id','name'])

,fdc_id,data_type,description,food_category_id,nutrient_value,name,unit_name
5702702,167512.0,sr_legacy_food,"Pillsbury Golden Layer Buttermilk Biscuits, Ar...",18,307.00000,kcal3,KCAL
7616965,167512.0,sr_legacy_food,"Pillsbury Golden Layer Buttermilk Biscuits, Ar...",18,1286.00000,kcal3,kJ
5702703,167513.0,sr_legacy_food,"Pillsbury, Cinnamon Rolls with Icing, refriger...",18,330.00000,kcal3,KCAL
7616966,167513.0,sr_legacy_food,"Pillsbury, Cinnamon Rolls with Icing, refriger...",18,1381.00000,kcal3,kJ
5702704,167514.0,sr_legacy_food,"Kraft Foods, Shake N Bake Original Recipe, Coa...",18,377.00000,kcal3,KCAL
7616967,167514.0,sr_legacy_food,"Kraft Foods, Shake N Bake Original Recipe, Coa...",18,1577.00000,kcal3,kJ
5702705,167515.0,sr_legacy_food,"George Weston Bakeries, Thomas English Muffins",18,232.00000,kcal3,KCAL
7616968,167515.0,sr_legacy_food,"George Weston Bakeries, Thomas English Muffins",18,972.00000,kcal3,kJ
5702706,167516.0,sr_legacy_food,"Waffles, buttermilk, frozen, ready-to-heat",18,273.00000,kcal3,KCAL
7616969,167516.0,sr_legacy_food,"Waffles, buttermilk, frozen, ready-to-heat",18,1144.00000,kcal3,kJ


In [12]:
df['nameunit'] = df['name'] + '_' + df['unit_name']
df = df.drop(['unit_name','name'], axis = 1)
df['nameunit'].sort_values().unique()

array(['ALA_G', 'ARA_G', 'B12_UG', 'B1_MG', 'B2_MG', 'B3_MG', 'B5_MG',
       'B6_MG', 'B7_UG', 'B9_UG', 'B_UG', 'CHO1_G', 'CHO2_G',
       'CHOfiber1_G', 'CHOfiber2_G', 'CHOstarch_G', 'CHOsugar_G', 'Ca_MG',
       'Cl_MG', 'Co_UG', 'Cr_UG', 'Cu_MG', 'DHA_G', 'DPA_G', 'EPA_G',
       'F_UG', 'Fe_MG', 'I_UG', 'K_MG', 'MUFA_G', 'Mg_MG', 'Mn_MG',
       'Mo_UG', 'Na_MG', 'Ni_UG', 'PUFA_G', 'P_MG', 'S_MG', 'Se_UG',
       'Zn_MG', 'alanine_G', 'alcohol_G', 'alpha_tocopherol_MG',
       'arginine_G', 'aspartate_G', 'caffeine_MG', 'cholesterol_MG',
       'cysteine_G', 'cystine_G', 'fructose_G', 'glucose_G',
       'glutamate_G', 'glycine_G', 'histidine_G', 'hydroxyproline_G',
       'insol_fiber_G', 'isoleucine_G', 'kcal1_KCAL', 'kcal2_KCAL',
       'kcal3_KCAL', 'kcal3_kJ', 'lactose_G', 'leucine_G', 'lycopene_UG',
       'lysine_G', 'maltose_G', 'methionine_G', 'phenylalanine_G',
       'proline_G', 'protein_G', 'pufa_18_2_n6_G', 'pufa_18_3_n6_G',
       'pufa_20_2_n6_G', 'pufa_20_3_n3_G',

In [13]:
# round to 3 sig figs if duplicated, then remove
df['description'] = df['description'].str.lower()
subs = df.duplicated(subset=['fdc_id','description','nameunit'], keep=False)
df.loc[subs, 'nutrient_value'] = df.loc[subs, 'nutrient_value'].apply(lambda x: round(x, sigfigs = 3))
df.drop_duplicates(inplace = True)
# the remaining duplicates have slight differences in 'nutrient_value'
df[df.duplicated(subset=['fdc_id','description','nameunit'],keep=False)].sort_values(by = ['fdc_id','nameunit'])

/tmp/ipykernel_1103/1453687956.py:4: UserWarning: 3 significant figures requested from number with only 2 significant figures
  df.loc[subs, 'nutrient_value'] = df.loc[subs, 'nutrient_value'].apply(lambda x: round(x, sigfigs = 3))
/tmp/ipykernel_1103/1453687956.py:4: UserWarning: 3 significant figures requested from number with only 1 significant figures
  df.loc[subs, 'nutrient_value'] = df.loc[subs, 'nutrient_value'].apply(lambda x: round(x, sigfigs = 3))


,fdc_id,data_type,description,food_category_id,nutrient_value,nameunit
12447392,321358.0,foundation_food,"hummus, commercial",16,71.1,Mg_MG
27144419,321358.0,foundation_food,"hummus, commercial",16,71.0,Mg_MG
5710495,321358.0,foundation_food,"hummus, commercial",16,229.0,kcal3_KCAL
27110433,321358.0,foundation_food,"hummus, commercial",16,243.0,kcal3_KCAL
12447394,321360.0,foundation_food,"tomatoes, grape, raw",11,11.9,Mg_MG
27145130,321360.0,foundation_food,"tomatoes, grape, raw",11,12.0,Mg_MG
5710497,321360.0,foundation_food,"tomatoes, grape, raw",11,27.0,kcal3_KCAL
27111144,321360.0,foundation_food,"tomatoes, grape, raw",11,31.0,kcal3_KCAL
12447432,321611.0,foundation_food,"beans, snap, green, canned, regular pack, drai...",11,12.7,Mg_MG
27143935,321611.0,foundation_food,"beans, snap, green, canned, regular pack, drai...",11,13.0,Mg_MG


In [14]:
# round to 2 sig figs if duplicated, then remove
subs = df.duplicated(subset=['fdc_id','description','nameunit'], keep=False)
df.loc[subs, 'nutrient_value'] = df.loc[subs, 'nutrient_value'].apply(lambda x: round(x, sigfigs = 2))
df.drop_duplicates(inplace = True)

df[df.duplicated(subset=['fdc_id','description','nameunit'],keep=False)].sort_values(by = ['fdc_id','nameunit'])

/tmp/ipykernel_1103/185672087.py:3: UserWarning: 2 significant figures requested from number with only 1 significant figures
  df.loc[subs, 'nutrient_value'] = df.loc[subs, 'nutrient_value'].apply(lambda x: round(x, sigfigs = 2))


,fdc_id,data_type,description,food_category_id,nutrient_value,nameunit
5710495,321358.0,foundation_food,"hummus, commercial",16,230.000,kcal3_KCAL
27110433,321358.0,foundation_food,"hummus, commercial",16,240.000,kcal3_KCAL
5710497,321360.0,foundation_food,"tomatoes, grape, raw",11,27.000,kcal3_KCAL
27111144,321360.0,foundation_food,"tomatoes, grape, raw",11,31.000,kcal3_KCAL
5710498,321611.0,foundation_food,"beans, snap, green, canned, regular pack, drai...",11,21.000,kcal3_KCAL
27109949,321611.0,foundation_food,"beans, snap, green, canned, regular pack, drai...",11,24.000,kcal3_KCAL
5710506,323505.0,foundation_food,"kale, raw",11,35.000,kcal3_KCAL
27109993,323505.0,foundation_food,"kale, raw",11,43.000,kcal3_KCAL
12447662,324653.0,foundation_food,"pickles, cucumber, dill or kosher dill",11,7.100,Mg_MG
27144082,324653.0,foundation_food,"pickles, cucumber, dill or kosher dill",11,7.000,Mg_MG


In [15]:
# won't take the remaining duplicates down to 1 sig fig
# instead, I'll drop the second row
df.drop_duplicates(subset=['fdc_id','description','nameunit'], inplace = True)
df.shape

(24191496, 6)

In [16]:
df = df.pivot(index = ['fdc_id','description','food_category_id','data_type'],
              columns = 'nameunit',
              values = 'nutrient_value')
df

nameunit                                                                                       ALA_G  \
fdc_id    description                                        food_category_id data_type                
NaN       almond butter, creamy                              NaN              NaN                NaN   
          apples, fuji, with skin, raw                       NaN              NaN                NaN   
          apples, gala, with skin, raw                       NaN              NaN                NaN   
          apples, granny smith, with skin, raw               NaN              NaN                NaN   
          apples, honeycrisp, with skin, raw                 NaN              NaN                NaN   
          apples, red delicious, with skin, raw              NaN              NaN                NaN   
          applesauce, unsweetened, with added vitamin c      NaN              NaN                NaN   
          babyfood, cereal, barley, dry fortified            NaN              NaN                NaN   
          babyfood, cereal, oatmeal, dry fortified           NaN              NaN                NaN   
          babyfood, cereal, rice, dry fortified              NaN              NaN                NaN   
          babyfood, gerber, graduates lil biscuits vanill... NaN              NaN                NaN   
          babyfood, juice, apple                             NaN              NaN                NaN   
          babyfood, multigrain whole grain cereal, dry fo... NaN              NaN                NaN   
          bagels, plain, enriched, with calcium propionat... NaN              NaN                NaN   
          bagels, wheat                                      NaN              NaN                NaN   
...                                                                                              ...   
2751489.0 green onion, (scallion), bulb and greens, root ... NaN              sub_sample_food    NaN   
2751490.0 green onion, (scallion), bulb and greens, root ... NaN              sub_sample_food    NaN   
2751491.0 green onion, (scallion), bulb and greens, root ... NaN              sub_sample_food    NaN   
2751492.0 green onion, (scallion), bulb and greens, root ... NaN              sub_sample_food    NaN   
2751493.0 green onion, (scallion), bulb and greens, root ... NaN              sub_sample_food    NaN   
2751494.0 green onion, (scallion), bulb and greens, root ... NaN              sub_sample_food    NaN   
2751495.0 green onion, (scallion), bulb and greens, root ... NaN              sub_sample_food    NaN   
2751496.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   
2751497.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   
2751498.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   
2751499.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   
2751500.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   
2751501.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   
2751502.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   
2751503.0 shallots, bulb, peeled, root removed, raw          NaN              sub_sample_food    NaN   

nameunit                                                                                       ARA_G  \
fdc_id    description                                        food_category_id data_type                
NaN       almond butter, creamy                              NaN              NaN                NaN   
          apples, fuji, with skin, raw                       NaN              NaN                NaN   
          apples, gala, with skin, raw                       NaN              NaN                NaN   
          apples, granny smith, with skin, raw 

In [17]:
df.columns.sort_values()

Index(['ALA_G', 'ARA_G', 'B12_UG', 'B1_MG', 'B2_MG', 'B3_MG', 'B5_MG', 'B6_MG',
       'B7_UG', 'B9_UG', 'B_UG', 'CHO1_G', 'CHO2_G', 'CHOfiber1_G',
       'CHOfiber2_G', 'CHOstarch_G', 'CHOsugar_G', 'Ca_MG', 'Cl_MG', 'Co_UG',
       'Cr_UG', 'Cu_MG', 'DHA_G', 'DPA_G', 'EPA_G', 'F_UG', 'Fe_MG', 'I_UG',
       'K_MG', 'MUFA_G', 'Mg_MG', 'Mn_MG', 'Mo_UG', 'Na_MG', 'Ni_UG', 'PUFA_G',
       'P_MG', 'S_MG', 'Se_UG', 'Zn_MG', 'alanine_G', 'alcohol_G',
       'alpha_tocopherol_MG', 'arginine_G', 'aspartate_G', 'caffeine_MG',
       'cholesterol_MG', 'cysteine_G', 'cystine_G', 'fructose_G', 'glucose_G',
       'glutamate_G', 'glycine_G', 'histidine_G', 'hydroxyproline_G',
       'insol_fiber_G', 'isoleucine_G', 'kcal1_KCAL', 'kcal2_KCAL',
       'kcal3_KCAL', 'kcal3_kJ', 'lactose_G', 'leucine_G', 'lycopene_UG',
       'lysine_G', 'maltose_G', 'methionine_G', 'phenylalanine_G', 'proline_G',
       'protein_G', 'pufa_18_2_n6_G', 'pufa_18_3_n6_G', 'pufa_20_2_n6_G',
       'pufa_20_3_n3_G', 'pufa_

In [18]:
# condense duplicated nutrients
# condense the many vit E columns
vit_E = ['alpha_tocopherol_MG','vit_e_MG','vit_e_MG_ATE']
df['total_vit_e_mg'] = df[vit_E].sum(axis = 1, skipna = True)
df.loc[df[vit_E].isna().sum(axis=1) == len(vit_E), 'total_vit_e_mg'] = np.nan
df.drop(vit_E, axis = 1, inplace = True)

In [19]:
# condense carbohydrates
df.loc[df['CHO1_G'].isna(), 'CHO1_G'] = df.loc[df['CHO1_G'].isna(), 'CHO2_G']
df.drop('CHO2_G', axis = 1, inplace = True)

# condense fiber columns
df.loc[df['CHOfiber1_G'].isna(), 'CHOfiber1_G'] = df.loc[df['CHOfiber1_G'].isna(), 'CHOfiber2_G']
df.drop('CHOfiber2_G', axis = 1, inplace = True)

# condense energy columns
df.loc[df['kcal1_KCAL'].isna(), 'kcal1_KCAL'] = df.loc[df['kcal1_KCAL'].isna(), 'kcal2_KCAL']
df.drop('kcal2_KCAL', axis = 1, inplace = True)
df.loc[df['kcal1_KCAL'].isna(), 'kcal1_KCAL'] = df.loc[df['kcal1_KCAL'].isna(), 'kcal3_KCAL']
df.drop(['kcal3_KCAL','kcal3_kJ'], axis = 1, inplace = True)

# condense total fat columns
df.loc[df['total_fat_G'].isna(), 'total_fat_G'] = df.loc[df['total_fat_G'].isna(), 'total_fat2_G']
df.drop('total_fat2_G', axis = 1, inplace = True)

# add up vitamin K columns
ks = ['vit_k_d_UG', 'vit_k_m_UG', 'vit_k_p_UG']
df['vit_k_UG'] = df[ks].sum(axis = 1)
df.loc[df[ks].isna().sum(axis=1) == len(ks), 'vit_k_UG'] = np.nan
df.drop(ks, axis = 1, inplace=True)

In [20]:
# many rows are the same food with different fdc_id
df = df.reset_index(names=['fdc_id', 'description','category','data_type'])
(df.loc[df.duplicated(subset=['description'],keep=False),
 ['fdc_id','description','category','data_type']]
 .sort_values(by = ['description']))

nameunit,fdc_id,description,category,data_type
1948417,2742400.0,advancepierre flamebroiled rib shaped pork pa...,Meat/Poultry/Other Animals - Prepared/Processed,branded_food
1943455,2737438.0,advancepierre flamebroiled rib shaped pork pa...,Meat/Poultry/Other Animals - Prepared/Processed,branded_food
1908359,2700133.0,advancepierre flamebroiled rib shaped pork pa...,Meat/Poultry/Other Animals - Prepared/Processed,branded_food
1940278,2734261.0,advancepierre flamebroiled rib shaped pork pa...,Meat/Poultry/Other Animals - Prepared/Processed,branded_food
1782528,2571794.0,all natural gluten free chicken nuggets,"Frozen Poultry, Chicken & Turkey",branded_food
1744193,2533357.0,all natural gluten free chicken nuggets,"Frozen Poultry, Chicken & Turkey",branded_food
1827653,2617100.0,all natural gluten free chicken nuggets,"Frozen Poultry, Chicken & Turkey",branded_food
1815065,2604504.0,all natural rosemary & olive oil basmati rice...,Flavored Rice Dishes,branded_food
1793660,2583001.0,all natural rosemary & olive oil basmati rice...,Flavored Rice Dishes,branded_food
875282,1568995.0,almond milk,Plant Based Milk,branded_food


In [21]:
# coalesce all non-missing values from the duplicates into one row.
# takes > 20 mins
df = df.drop('fdc_id', axis = 1)

id_cols = ['description']          # columns defining duplicates
value_cols = df.columns.difference(id_cols)

def first_non_null(s):
    return s.dropna().iloc[0] if s.notna().any() else pd.NA

df = (
    df.groupby(id_cols, as_index=False)
      .agg({col: first_non_null for col in value_cols})
)

df

nameunit,description,ALA_G,ARA_G,B12_UG,B1_MG,B2_MG,B3_MG,B5_MG,B6_MG,B7_UG,B9_UG,B_UG,CHO1_G,CHOfiber1_G,CHOstarch_G,CHOsugar_G,Ca_MG,Cl_MG,Co_UG,Cr_UG,Cu_MG,DHA_G,DPA_G,EPA_G,F_UG,Fe_MG,I_UG,K_MG,MUFA_G,Mg_MG,Mn_MG,Mo_UG,Na_MG,Ni_UG,PUFA_G,P_MG,S_MG,Se_UG,Zn_MG,alanine_G,alcohol_G,arginine_G,aspartate_G,caffeine_MG,category,cholesterol_MG,cysteine_G,cystine_G,data_type,fructose_G,glucose_G,glutamate_G,glycine_G,histidine_G,hydroxyproline_G,insol_fiber_G,isoleucine_G,kcal1_KCAL,lactose_G,leucine_G,lycopene_UG,lysine_G,maltose_G,methionine_G,phenylalanine_G,proline_G,protein_G,pufa_18_2_n6_G,pufa_18_3_n6_G,pufa_20_2_n6_G,pufa_20_3_n3_G,pufa_20_3_n6_G,pufa_20_4_G,sat_fat_G,serine_G,sol_fiber_G,sucrose_G,taurine_G,threonine_G,total_choline_MG,total_fat_G,total_vit_d_IU,total_vit_d_UG,total_vit_e_mg,trans_fat_G,tryptophan_G,tyrosine_G,valine_G,vit_a_IU,vit_c_MG,vit_k_UG,water_G
0,advancepierre flamebroiled rib shaped pork pa...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.17,1.0,<NA>,<NA>,41.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,288.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Meat/Poultry/Other Animals - Prepared/Processed,45.0,<NA>,<NA>,branded_food,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,209.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14.77,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4.13,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,12.1,<NA>,<NA>,<NA>,0.09,<NA>,<NA>,<NA>,<NA>,1.8,<NA>,<NA>
1,all natural gluten free chicken nuggets,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.24,0.0,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.85,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,400.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"Frozen Poultry, Chicken & Turkey",47.0,<NA>,<NA>,branded_food,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,194.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18.82,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2.35,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,9.41,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,0.0,1.4,<NA>,<NA>
2,all natural rosemary & olive oil basmati rice...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,78.26,2.2,<NA>,<NA>,22.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.04,<NA>,87.0,<NA>,<NA>,<NA>,<NA>,1000.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Flavored Rice Dishes,0.0,<NA>,<NA>,branded_food,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,326.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6.52,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,0.0,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,almond milk,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.33,0.4,<NA>,<NA>,42.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.15,<NA>,75.0,0.62,<NA>,<NA>,<NA>,62.0,<NA>,0.21,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Plant Based Milk,0.0,<NA>,<NA>,branded_food,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,25.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.42,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.04,42.0,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,208.0,0.0,<NA>,<NA>
4,"almond milk, original",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3.33,0.4,<NA>,<NA>,42.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.15,<NA>,75.0,0.62,<NA>,<NA>,<NA>,62.0,<NA>,0.21,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Plant Based Milk,0.0,<NA>,<NA>,branded_food,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,25.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.42,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.04,42.0,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,208.0,0.0,<NA>,<NA>
5,artisan smooth & creamy gelato,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,25.47,4.7,<NA>,<NA>,123.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.17,<NA>,170.0,<NA>,<NA>,<NA>,<NA>,52.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Ice Cream & Frozen Yogurt,14.0,<NA>,<NA>,branded_food,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,198.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4.72,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.49,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.38,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,0.9,<N

In [22]:
df.to_parquet(data_path + 'FDCeverything.parquet.gzip',compression = 'gzip')
